In [25]:
import sys
sys.path.append('..')

import os
from sqlalchemy import create_engine, text
import geopandas as gpd
import pandas as pd

import numpy as np
from shapely.geometry import shape
from geoalchemy2 import Geometry, WKTElement

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import API_AMIGOCLOUD_TOKEN_ADM
from config import POSTGRES_UTEA

In [26]:
def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

# get catastro desde base de datos
def get_catastro():
    engine = obtener_engine()
    try:
        query = f'''
            SELECT * FROM catastro_iag.catastro
        '''
        gdf = gpd.read_postgis(query, engine, geom_col='geom')
        return gdf
    except Exception as e:
        print(f"❌ Error al obtener la capa de catastro: {e}")
        return gpd.GeoDataFrame()
    return None


In [27]:
gdf_catrastro = get_catastro()
# filtar caña habilitada para para zafra actual
gdf_catastro_canha = gdf_catrastro[(gdf_catrastro['unidad_03'] != 0) & (gdf_catrastro['cultivo'] == 'canha') & (gdf_catrastro['zafra'] == 2026)]
gdf_catastro_canha['area'].sum()

58710.5

In [29]:
# seleccionar columnas necesarias
gdf_catastro_canha = gdf_catastro_canha[['idd', 'geom', 'unidad_01', 'unidad_02', 'unidad_03', 'unidad_04', 'unidad_05', 'area']]
gdf_catastro_canha.head(3)

,idd,geom,unidad_01,unidad_02,unidad_03,unidad_04,unidad_05,area
2,10009,"MULTIPOLYGON (((506528.416 8095647.843, 506537...",32.0,RIO PAILON--ROCA GIL VICENTE,633.0,AGROP. VICENTE ROCA GIL SRL,L2,9.80
3,10080,"MULTIPOLYGON (((506006.724 8095047.828, 506003...",32.0,RIO PAILON--ROCA GIL VICENTE,633.0,AGROP. VICENTE ROCA GIL SRL,L4,12.80
4,15905,"MULTIPOLYGON (((506095.082 8094940.498, 506068...",32.0,RIO PAILON--ROCA GIL VICENTE,633.0,AGROP. VICENTE ROCA GIL SRL,L3,10.61


In [30]:
gdf_catastro_canha['unidad_01'] = gdf_catastro_canha['unidad_01'].astype(int)
# Crear la nueva columna concatenada
gdf_catastro_canha['id_compuesto'] = (
    gdf_catastro_canha['unidad_01'].astype(str) + 
    '|' + 
    gdf_catastro_canha['unidad_05'].astype(str)
)
ids = gdf_catastro_canha[['id_compuesto']]

In [31]:
# 1. Contar las ocurrencias de cada id_compuesto
# value_counts() devuelve una serie con el ID como índice y la cuenta como valor
counts = gdf_catastro_canha['id_compuesto'].value_counts().reset_index()

# 2. Renombrar las columnas para que sea claro
counts.columns = ['id_compuesto', 'cantidad']

# 3. Filtrar para quedarte solo con los que se repiten (cantidad > 1)
duplicados = counts[counts['cantidad'] > 1]

In [32]:
duplicados

,id_compuesto,cantidad
